# Anotacoes

**Regras:**
- % defeito = Qtd Defeito / Qtd Amostra (corrigido)

**Pontos a entender:**

-  Nos top 5 desvios, há siglas presentes em AQ que na AF não há
    - Isso pode ser que a AF detectou siglas mais fortes do que a AQ detectou
    - Como estamos vendo em top 5, eles podem não estar aparecendo em AF devido isso
    - Lembrando: AF é a referencia deles, apenas usam dados da AF


# Importando bibliotecas

In [68]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import ipywidgets as widgets
from IPython.display import display

from plotly.subplots import make_subplots

# Importando dados

In [69]:
df_hh = pd.read_excel('../data/raw/Desvios por máquina - HH.xlsx', header=2)

df_diario = pd.read_excel('../data/raw/Desvios por máquina - Completo.xlsx', header=2)

c:\Users\evosystem03.ti\Documents\Demanda Carteira\Projeto\predicao-carteira-wheaton\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\evosystem03.ti\Documents\Demanda Carteira\Projeto\predicao-carteira-wheaton\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


# Visualizando dados

In [70]:
df_hh

,Data wht (dia) amostra,Forno,Maquina,Prefixo,OP Vertech,NQI (amostra),Hora a Hora Wht,Localizacao,Desvio sigla,Qtd Amostra (corrigido),Qtd Defeito,% defeito
0,2026-08-10,A,A1,SB -1035-SWN,198618,F,7,AQ,BOL,12,1,0.083333
1,2026-08-10,A,A1,SB -1035-SWN,198618,F,7,AF,BOL,24,3,0.125000
2,2026-08-10,A,A1,SB -1035-SWN,198618,F,8,AQ,BOL,12,1,0.083333
3,2026-08-10,A,A1,SB -1035-SWN,198618,F,8,AQ,DOB,12,1,0.083333
4,2026-08-10,A,A1,SB -1035-SWN,198618,F,8,AF,BOL,34,12,0.352941
...,...,...,...,...,...,...,...,...,...,...,...,...
2094,2026-08-10,1,11,LB -0586-S,198660,B,6,AF,ATR,8,1,0.125000
2095,2026-08-10,1,11,LB -0586-S,198660,B,6,AF,FFU,8,2,0.250000
2096,2026-08-10,1,11,LB -0586-S,198660,B,6,AF,RAB,8,1,0.125000
2097,2026-08-10,1,11,LB -0586-S,198660,B,6,CF,FFU,250,12,0.048000


In [71]:
df_diario

,Data wht (dia) amostra,Forno,Maquina,Prefixo,OP Vertech,NQI (amostra),Localizacao,Desvio sigla,Qtd Amostra (corrigido),Qtd Defeito,% defeito
0,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,BOL,252,42,0.166667
1,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,COS,12,1,0.083333
2,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,DOB,72,6,0.083333
3,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,FSE,12,3,0.250000
4,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,PED,36,6,0.166667
...,...,...,...,...,...,...,...,...,...,...,...
625,2026-08-10,1,11,LB -0586-S,198660,B,AF,TCT,40,7,0.175000
626,2026-08-10,1,11,LB -0586-S,198660,B,AF,TSU,32,4,0.125000
627,2026-08-10,1,11,LB -0586-S,198660,B,AF,TTO,16,3,0.187500
628,2026-08-10,1,11,LB -0586-S,198660,B,CF,FFU,375,18,0.048000


# Tratando dados

In [72]:
# visualizando variaveis e seus tipos
df_diario.info()

<class 'pandas.DataFrame'>
RangeIndex: 630 entries, 0 to 629
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Data wht (dia) amostra   630 non-null    datetime64[us]
 1   Forno                    630 non-null    str           
 2   Maquina                  630 non-null    str           
 3   Prefixo                  630 non-null    str           
 4   OP Vertech               630 non-null    str           
 5   NQI (amostra)            624 non-null    str           
 6   Localizacao              630 non-null    str           
 7   Desvio sigla             630 non-null    str           
 8   Qtd Amostra (corrigido)  630 non-null    int64         
 9   Qtd Defeito              630 non-null    int64         
 10  % defeito                630 non-null    float64       
dtypes: datetime64[us](1), float64(1), int64(2), str(7)
memory usage: 72.1 KB


In [73]:
# OPs que estamos trabalhando nas bases de métricas das máquinas em producoes
OPs = [
    '198594', '198660', '198592', '198176', '198456', '198618',
    '198698', '198567', '198135', '198659', '198639', '198645',
    '198701', '198704', '198707', '198694', '198658', '198663',
    '198613', '198693', '198546'
]

# Padroniza as OPs como texto sem espaços nos dados
df_diario['OP Vertech'] = (
    df_diario['OP Vertech']
    .astype(str)
    .str.strip()
)

# Padroniza os tipos de controle como texto sem espacos nos dados
df_diario['Desvio sigla'] = (
    df_diario['Desvio sigla']
    .astype(str)
    .str.strip()
)

# Filtra somente as OPs da nossa análise
df_diario = df_diario[
    df_diario['OP Vertech'].isin(OPs)
].copy()

display(df_diario.head(10))


,Data wht (dia) amostra,Forno,Maquina,Prefixo,OP Vertech,NQI (amostra),Localizacao,Desvio sigla,Qtd Amostra (corrigido),Qtd Defeito,% defeito
0,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,BOL,252,42,0.166667
1,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,COS,12,1,0.083333
2,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,DOB,72,6,0.083333
3,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,FSE,12,3,0.250000
4,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,PED,36,6,0.166667
5,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,PIP,24,2,0.083333
6,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,SUO,12,1,0.083333
7,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,T -,12,1,0.083333
8,2026-08-10,A,A1,SB -1035-SWN,198618,F,AF,BOA,240,17,0.070833
9,2026-08-10,A,A1,SB -1035-SWN,198618,F,AF,BOL,428,62,0.144860


# Analises

## Maiores desvios por producao

In [74]:
# agrupando desvios (Desvio sigla) por OP Vertech e Localizacao, somando a quantidade de desvios (Qtd. Desvios) e a porcentagem de defeito (% defeito)
df_grouped = df_diario.groupby(['OP Vertech', 'Desvio sigla', 'Localizacao']).agg({
    'Qtd Amostra (corrigido)': 'sum',
    '% defeito': 'sum' # Pode ser mean?
}).reset_index()

display(df_grouped)

,OP Vertech,Desvio sigla,Localizacao,Qtd Amostra (corrigido),% defeito
0,198135,A -,AF,10,0.200000
1,198135,ABO,AF,24,0.041667
2,198135,ADE,AF,24,0.041667
3,198135,BC-,AF,24,0.083333
4,198135,BOA,AF,72,0.041667
...,...,...,...,...,...
597,198707,RAB,AF,48,0.104167
598,198707,RCO,AF,24,0.125000
599,198707,RGA,AF,12,0.083333
600,198707,SUO,AF,108,0.148148


In [75]:
# filtrando uma OP e sua localizacao para somar a % defeito
df_filtered = df_grouped[
    (df_grouped['OP Vertech'] == '198135') &
    (df_grouped['Localizacao'] == 'AF')
]

# somando a % defeito para a OP 198135 e Localizacao AF
total_defect_percentage = df_filtered['% defeito'].sum()

print(f"Percentual total de defeitos da OP 198135 e Localizacao AF: {total_defect_percentage.round(2)} %")

Percentual total de defeitos da OP 198135 e Localizacao AF: 2.13 %


In [76]:
# Remove nulos essenciais e ordena tudo apenas uma vez
df_plot = (
    df_grouped
    .dropna(subset=['OP Vertech', 'Localizacao'])
    .sort_values(
        ['OP Vertech', 'Localizacao', '% defeito'],
        ascending=[True, True, False]
    )
)

# Um gráfico para cada OP Vertech
for op_vertech, df_op in df_plot.groupby('OP Vertech', sort=False):

    # Separa os dados por Localizacao
    dados = {
        str(localizacao): df_local
        for localizacao, df_local
        in df_op.groupby('Localizacao', sort=False)
    }

    if not dados:
        continue

    # Primeira localização exibida
    local_inicial = next(iter(dados))
    df_inicial = dados[local_inicial]

    # Cria o gráfico
    fig = go.Figure(
        go.Bar(
            x=df_inicial['Desvio sigla'],
            y=df_inicial['% defeito'],

            # Texto exibido acima das barras
            text=df_inicial['% defeito'],
            texttemplate='%{y:.2%}',
            textposition='outside',

            cliponaxis=False,

            # Hover
            hovertemplate=(
                '<b>Desvio:</b> %{x}<br>'
                '<b>Defeito:</b> %{y:.2%}'
                '<extra></extra>'
            )
        )
    )

    # Dropdown de Localizacao
    botoes = [
        dict(
            label=localizacao,
            method='update',
            args=[
                {
                    'x': [df_local['Desvio sigla']],
                    'y': [df_local['% defeito']],
                    'text': [df_local['% defeito']]
                },
                {
                    'title.text': (
                        f'Percentual de Defeitos - OP Vertech: {op_vertech}'
                        f'<br>Localização: {localizacao}'
                    )
                }
            ]
        )
        for localizacao, df_local in dados.items()
    ]

    # Layout
    fig.update_layout(
        title=dict(
            text=(
                f'Percentual de Desvios - OP Vertech: {op_vertech}'
                f'<br>Localização: {local_inicial}'
            ),
            x=0.5
        ),

        # Eixo X
        xaxis=dict(
            title='Tipo de Desvio'
        ),

        # Eixo Y formatado como percentual
        yaxis=dict(
            title='Percentual de Desvios (%)',
            tickformat='.0%'  # 0.10 -> 10%
        ),

        height=600,
        showlegend=False,
        bargap=0.2,

        margin=dict(
            t=130
        ),

        # Dropdown
        updatemenus=[
            dict(
                buttons=botoes,
                direction='down',
                showactive=True,
                x=0,
                y=1.15,
                xanchor='left',
                yanchor='top'
            )
        ],

        # Texto "Localização:"
        annotations=[
            dict(
                text='<b>Localização:</b>',
                x=0,
                y=1.22,
                xref='paper',
                yref='paper',
                showarrow=False,
                xanchor='left'
            )
        ]
    )

    fig.show()

## Top desvios por OP da localizacao AF

In [77]:
df_grouped_AF = df_grouped[df_grouped['Localizacao'] == 'AF'].copy()

# Remove nulos essenciais e ordena
df_plot_AF = (
    df_grouped_AF
    .dropna(subset=['OP Vertech'])
    .sort_values(
        ['OP Vertech', '% defeito'],
        ascending=[True, False]
    )
)

# Um gráfico para cada OP Vertech
for op_vertech, df_op in df_plot_AF.groupby('OP Vertech', sort=False):

    if df_op.empty:
        continue

    # Maior valor SOMENTE desta OP
    valor_max = df_op['% defeito'].max()

    # Cria o gráfico
    fig = go.Figure(
        go.Bar(
            x=df_op['Desvio sigla'],
            y=df_op['% defeito'],

            # Cor gradual dentro de cada gráfico
            marker=dict(
                color=df_op['% defeito'],
                colorscale='Reds',
                cmin=0,
                cmax=valor_max,
                showscale=False
            ),

            # Texto exibido acima das barras
            text=df_op['% defeito'],
            texttemplate='%{y:.2%}',
            textposition='outside',

            cliponaxis=False,

            # Hover
            hovertemplate=(
                '<b>Desvio:</b> %{x}<br>'
                '<b>Defeito:</b> %{y:.2%}'
                '<extra></extra>'
            )
        )
    )

    # Layout
    fig.update_layout(
        title=dict(
            text=(
                f'Percentual de Desvios - OP Vertech: {op_vertech}'
                f'<br>Localização: AF'
            ),
            x=0.5
        ),

        xaxis=dict(
            title='Tipo de Desvio'
        ),

        yaxis=dict(
            title='Percentual de Desvios (%)',
            tickformat='.0%'
        ),

        height=600,
        showlegend=False,
        bargap=0.2,

        margin=dict(
            t=100
        )
    )

    fig.show()

In [78]:
'''# Remove OPs nulas, ordena do maior para o menor percentual
# dentro de cada OP e mantém somente o TOP 5 de cada OP.
df_plot_AF = (
    df_grouped_AF
    .dropna(subset=['OP Vertech'])
    .sort_values(
        ['OP Vertech', '% defeito'],
        ascending=[True, False]
    )
    .groupby(
        'OP Vertech',
        sort=False,
        observed=True
    )
    .head(5)
)

hover_template = (
    '<b>Desvio:</b> %{x}<br>'
    '<b>Defeito:</b> %{y:.2%}'
    '<extra></extra>'
)

layout_base = dict(
    xaxis=dict(
        title='Tipo de Desvio'
    ),

    yaxis=dict(
        title='Percentual de Defeitos (%)',
        tickformat='.0%'
    ),

    height=600,
    showlegend=False,
    bargap=0.2,

    margin=dict(
        t=100
    )
)

for op_vertech, df_op in df_plot_AF.groupby(
    'OP Vertech',
    sort=False,
    observed=True
):

    # Converte apenas as colunas utilizadas para arrays
    desvios = df_op['Desvio sigla'].to_numpy(copy=False)
    defeitos = df_op['% defeito'].to_numpy(copy=False)

    # Maior percentual desta OP
    valor_max = defeitos.max()

    # Cria o gráfico
    fig = go.Figure(
        go.Bar(
            x=desvios,
            y=defeitos,

            # Cor proporcional ao percentual
            marker=dict(
                color=defeitos,
                colorscale='Reds',
                cmin=0,
                cmax=valor_max,
                showscale=False
            ),

            # Texto acima das barras
            texttemplate='%{y:.2%}',
            textposition='outside',

            cliponaxis=False,

            # Hover
            hovertemplate=hover_template
        )
    )

    # Layout
    fig.update_layout(
        **layout_base,

        title=dict(
            text=(
                f'Top 5 Percentual de Defeitos - OP Vertech: {op_vertech}'
                '<br>Localização: AF'
            ),
            x=0.5
        )
    )

    fig.show()'''


# Remove OPs nulas, ordena do maior para o menor percentual
# dentro de cada OP e mantém somente o TOP 5 de cada OP.
df_plot_AF = (
    df_grouped_AF
    .dropna(subset=["OP Vertech"])
    .sort_values(
        ["OP Vertech", "% defeito"],
        ascending=[True, False],
    )
    .groupby(
        "OP Vertech",
        sort=False,
        observed=True,
    )
    .head(5)
)

hover_template = (
    "<b>Desvio:</b> %{y}<br>"
    "<b>Defeito:</b> %{x:.2%}"
    "<extra></extra>"
)

layout_base = dict(
    xaxis=dict(
        title="Percentual de Defeitos (%)",
        tickformat=".0%",
    ),
    yaxis=dict(
        title="Tipo de Desvio",
        autorange="reversed",
    ),
    width=600,
    height=520,
    showlegend=False,
    bargap=0.2,
    margin=dict(
        l=100,
        r=40,
        t=85,
        b=60,
    ),
)

for op_vertech, df_op in df_plot_AF.groupby(
    "OP Vertech",
    sort=False,
    observed=True,
):
    # Converte as colunas utilizadas para arrays
    desvios = df_op["Desvio sigla"].to_numpy(copy=False)
    defeitos = df_op["% defeito"].to_numpy(copy=False)

    # Maior percentual desta OP
    valor_max = defeitos.max()

    # Cria o gráfico horizontal
    fig = go.Figure(
        go.Bar(
            x=defeitos,
            y=desvios,
            orientation="h",

            # Cor proporcional ao percentual
            marker=dict(
                color=defeitos,
                colorscale="Reds",
                cmin=0,
                cmax=valor_max,
                showscale=False,
            ),

            # Texto ao final das barras
            texttemplate="%{x:.2%}",
            textposition="inside",
            cliponaxis=False,

            # Hover
            hovertemplate=hover_template,
        )
    )

    fig.update_layout(
        **layout_base,
        title=dict(
            text=(
                f"Top 5 Percentual de Defeitos - "
                f"OP Vertech: {op_vertech}"
                "<br>Localização: AF"
            ),
            x=0.5,
        ),
    )

    fig.show()

## Comparação de cobertura de registros por localização

In [79]:
# Localizações que fazem parte da auditoria
LOCALIZACOES = ["AQ", "AF", "CF"]

cores_localizacoes = {
    "AQ": "#D62728",
    "AF": "#1F77B4",
    "CF": "#2CA02C",
}


# ============================================================
# 1. Preparação da base
# ============================================================

base_auditoria = df_diario.copy()

for coluna in [
    "OP Vertech",
    "Localizacao",
    "Desvio sigla",
]:
    base_auditoria[coluna] = (
        base_auditoria[coluna]
        .astype("string")
        .str.strip()
    )

base_auditoria["Localizacao"] = (
    base_auditoria["Localizacao"]
    .str.upper()
)

for coluna in [
    "Qtd Amostra (corrigido)",
    "Qtd Defeito",
]:
    base_auditoria[coluna] = pd.to_numeric(
        base_auditoria[coluna],
        errors="coerce",
    )

base_auditoria = (
    base_auditoria[
        base_auditoria["Localizacao"].isin(
            LOCALIZACOES
        )
    ]
    .dropna(
        subset=[
            "OP Vertech",
            "Localizacao",
            "Desvio sigla",
        ]
    )
    .copy()
)


# ============================================================
# 2. Resumo por OP e localização
# ============================================================

resumo_op_localizacao = (
    base_auditoria
    .groupby(
        [
            "OP Vertech",
            "Localizacao",
        ],
        as_index=False,
        observed=True,
    )
    .agg(
        Quantidade_Registros=( # soma o total de registros por OP e Localizacao
            "Desvio sigla",
            "size",
        ),
        Quantidade_Desvios_Distintos=( # soma o total de desvios unicos por OP e Localizacao
            "Desvio sigla",
            "nunique",
        ),
        Soma_Amostras_por_Desvio=( # soma o total de "Qtd Amostra (corrigido)" por OP e Localizacao
            "Qtd Amostra (corrigido)",
            "sum",
        ),
        Quantidade_Total_Defeitos=( # soma o total de "Qtd Defeito" por OP e Localizacao
            "Qtd Defeito",
            "sum",
        ),
    )
)

# Não soma diretamente os percentuais.
# Calcula a proporção usando as quantidades agregadas.
# Percentual ponderado = (Quantidade total de defeitos / Soma das amostras informadas)
resumo_op_localizacao[
    "Percentual_Defeito_Ponderado"
] = (
    resumo_op_localizacao[
        "Quantidade_Total_Defeitos"
    ]
    / resumo_op_localizacao[
        "Soma_Amostras_por_Desvio"
    ].where(
        resumo_op_localizacao[
            "Soma_Amostras_por_Desvio"
        ].gt(0)
    )
)

display(resumo_op_localizacao)

,OP Vertech,Localizacao,Quantidade_Registros,Quantidade_Desvios_Distintos,Soma_Amostras_por_Desvio,Quantidade_Total_Defeitos,Percentual_Defeito_Ponderado
0,198135,AF,28,28,1970,139,0.070558
1,198135,AQ,12,12,660,73,0.110606
2,198176,AF,18,18,1168,174,0.148973
3,198176,AQ,8,8,224,56,0.250000
4,198176,CF,1,1,1200,0,0.000000
5,198456,AF,25,25,1030,169,0.164078
6,198456,AQ,12,12,130,24,0.184615
7,198456,CF,1,1,1625,0,0.000000
8,198546,AF,20,20,2032,192,0.094488
9,198546,AQ,11,11,480,61,0.127083


In [80]:
presenca_localizacoes = (
    pd.crosstab(
        base_auditoria["OP Vertech"],
        base_auditoria["Localizacao"],
    )
    .reindex(
        columns=LOCALIZACOES,
        fill_value=0,
    )
    .gt(0)
)

auditoria_cobertura_op = (
    presenca_localizacoes
    .rename(
        columns={
            localizacao:
                f"Possui_Registro_{localizacao}"
            for localizacao in LOCALIZACOES
        }
    )
)

auditoria_cobertura_op[
    "Quantidade_Localizacoes_Registradas"
] = presenca_localizacoes.sum(axis=1)

auditoria_cobertura_op[
    "Localizacoes_Ausentes"
] = [
    ", ".join(
        localizacao
        for localizacao in LOCALIZACOES
        if not linha[localizacao]
    ) or "Nenhuma"
    for _, linha in presenca_localizacoes.iterrows()
]

auditoria_cobertura_op = (
    auditoria_cobertura_op
    .reset_index()
)

display(auditoria_cobertura_op)

Localizacao,OP Vertech,Possui_Registro_AQ,Possui_Registro_AF,Possui_Registro_CF,Quantidade_Localizacoes_Registradas,Localizacoes_Ausentes
0,198135,True,True,False,2,CF
1,198176,True,True,True,3,Nenhuma
2,198456,True,True,True,3,Nenhuma
3,198546,True,True,True,3,Nenhuma
4,198567,True,True,False,2,CF
5,198592,True,True,True,3,Nenhuma
6,198594,True,True,True,3,Nenhuma
7,198613,True,True,False,2,CF
8,198618,True,True,False,2,CF
9,198639,True,True,False,2,CF


In [81]:
ops_com_localizacao_ausente = (
    auditoria_cobertura_op[
        auditoria_cobertura_op[
            "Localizacoes_Ausentes"
        ].ne("Nenhuma")
    ]
)

display(ops_com_localizacao_ausente)

Localizacao,OP Vertech,Possui_Registro_AQ,Possui_Registro_AF,Possui_Registro_CF,Quantidade_Localizacoes_Registradas,Localizacoes_Ausentes
0,198135,True,True,False,2,CF
4,198567,True,True,False,2,CF
7,198613,True,True,False,2,CF
8,198618,True,True,False,2,CF
9,198639,True,True,False,2,CF
10,198645,True,True,False,2,CF
12,198659,True,True,False,2,CF
14,198663,True,True,False,2,CF
15,198693,True,True,False,2,CF
17,198698,True,True,False,2,CF


In [82]:
total_ops = (
    base_auditoria["OP Vertech"].nunique()
)

resumo_cobertura_localizacao = (
    base_auditoria
    .groupby(
        "Localizacao",
        as_index=False,
        observed=True,
    )
    .agg(
        Quantidade_OPs_Registradas=(
            "OP Vertech",
            "nunique",
        ),
        Quantidade_Registros=(
            "Desvio sigla",
            "size",
        ),
        Quantidade_Desvios_Distintos=(
            "Desvio sigla",
            "nunique",
        ),
        Soma_Amostras_por_Desvio=(
            "Qtd Amostra (corrigido)",
            "sum",
        ),
        Quantidade_Total_Defeitos=(
            "Qtd Defeito",
            "sum",
        ),
    )
)

# calcula Cobertura = (Quantidade de OPs com registro na localização / Total de OPs) × 100
resumo_cobertura_localizacao[
    "Cobertura_OPs_Percentual"
] = (
    resumo_cobertura_localizacao[
        "Quantidade_OPs_Registradas"
    ]
    .div(total_ops)
    .mul(100)
)

resumo_cobertura_localizacao[
    "Percentual_Defeito_Ponderado"
] = (
    resumo_cobertura_localizacao[
        "Quantidade_Total_Defeitos"
    ]
    / resumo_cobertura_localizacao[
        "Soma_Amostras_por_Desvio"
    ].where(
        resumo_cobertura_localizacao[
            "Soma_Amostras_por_Desvio"
        ].gt(0)
    )
)

display(resumo_cobertura_localizacao)

,Localizacao,Quantidade_OPs_Registradas,Quantidade_Registros,Quantidade_Desvios_Distintos,Soma_Amostras_por_Desvio,Quantidade_Total_Defeitos,Cobertura_OPs_Percentual,Percentual_Defeito_Ponderado
0,AF,21,394,98,32671,3771,100.000000,0.115423
1,AQ,21,155,58,5795,823,100.000000,0.142019
2,CF,11,53,30,29685,166,52.380952,0.005592


In [83]:
fig = px.bar(
    resumo_cobertura_localizacao,
    x="Localizacao",
    y="Cobertura_OPs_Percentual",
    color="Localizacao",
    text="Cobertura_OPs_Percentual",
    color_discrete_map=cores_localizacoes,
    title="Cobertura de registros por localização",
    labels={
        "Localizacao": "Localização",
        "Cobertura_OPs_Percentual": "OPs com registro (%)",
    },
    hover_data={
        "Quantidade_OPs_Registradas": True,
        "Quantidade_Registros": True,
        "Quantidade_Desvios_Distintos": True,
        "Cobertura_OPs_Percentual": ":.2f",
    },
    template="plotly_white",
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
    cliponaxis=False,
)

fig.update_yaxes(
    range=[0, 105],
    ticksuffix="%",
)

fig.update_layout(
    title_x=0.5,
    showlegend=False,
    width=700,
    height=450,
)

fig.show()

In [84]:
fig = px.bar(
    resumo_op_localizacao,
    x="OP Vertech",
    y="Quantidade_Desvios_Distintos",
    color="Localizacao",
    barmode="group",
    color_discrete_map=cores_localizacoes,
    title="Quantidade de desvios distintos registrados por OP e localização",
    labels={
        "OP Vertech": "OP",
        "Quantidade_Desvios_Distintos": "Desvios distintos",
        "Localizacao": "Localização",
    },
    hover_data={
        "Quantidade_Registros": True,
        "Soma_Amostras_por_Desvio": ":,.0f",
        "Quantidade_Total_Defeitos": ":,.0f",
        "Percentual_Defeito_Ponderado": ":.2%",
    },
    template="plotly_white",
)

fig.update_layout(
    title_x=0.5,
    width=1100,
    height=550,
    xaxis_tickangle=-45,
    legend_title="Localização",
)

fig.show()

In [85]:
fig = px.bar(
    resumo_op_localizacao,
    x="OP Vertech",
    y="Percentual_Defeito_Ponderado",
    color="Localizacao",
    barmode="group",
    color_discrete_map=cores_localizacoes,
    title="Percentual ponderado de defeitos por OP e localização",
    labels={
        "OP Vertech": "OP",
        "Percentual_Defeito_Ponderado": "Percentual de defeitos",
        "Localizacao": "Localização",
    },
    hover_data={
        "Quantidade_Desvios_Distintos": True,
        "Soma_Amostras_por_Desvio": ":,.0f",
        "Quantidade_Total_Defeitos": ":,.0f",
        "Percentual_Defeito_Ponderado": ":.2%",
    },
    template="plotly_white",
)

fig.update_yaxes(
    tickformat=".1%",
)

fig.update_layout(
    title_x=0.5,
    width=1100,
    height=550,
    xaxis_tickangle=-45,
    legend_title="Localização",
)

fig.show()

## Comparação top desvios localizacao AQ vs. AF

In [86]:
# Cria a base utilizada na comparação entre AQ e AF
base_aq_af = df_diario.copy()

# Padroniza as colunas de texto
for coluna in [
    "OP Vertech",
    "Desvio sigla",
    "Localizacao",
]:
    base_aq_af[coluna] = (
        base_aq_af[coluna]
        .astype("string")
        .str.strip()
    )

base_aq_af["Localizacao"] = (
    base_aq_af["Localizacao"]
    .str.upper()
)

# Garante que as quantidades sejam numéricas
for coluna in [
    "Qtd Amostra (corrigido)",
    "Qtd Defeito",
]:
    base_aq_af[coluna] = pd.to_numeric(
        base_aq_af[coluna],
        errors="coerce",
    )

# Mantém apenas AQ e AF
base_aq_af = (
    base_aq_af[
        base_aq_af["Localizacao"].isin(
            ["AQ", "AF"]
        )
    ]
    .dropna(
        subset=[
            "OP Vertech",
            "Desvio sigla",
        ]
    )
    .copy()
)

# Consolida os valores por OP, desvio e localização
resumo_aq_af = (
    base_aq_af
    .groupby(
        [
            "OP Vertech",
            "Desvio sigla",
            "Localizacao",
        ],
        as_index=False,
        observed=True,
    )
    .agg(
        Qtd_Amostra=(
            "Qtd Amostra (corrigido)",
            "sum",
        ),
        Qtd_Defeito=(
            "Qtd Defeito",
            "sum",
        ),
    )
)

# Calcula o percentual consolidado de defeitos
resumo_aq_af["Percentual_Defeito"] = (
    resumo_aq_af["Qtd_Defeito"]
    / resumo_aq_af["Qtd_Amostra"].where(
        resumo_aq_af["Qtd_Amostra"].gt(0)
    )
)

display(resumo_aq_af.head())

,OP Vertech,Desvio sigla,Localizacao,Qtd_Amostra,Qtd_Defeito,Percentual_Defeito
0,198135,A -,AF,10,2,0.200000
1,198135,ABO,AF,24,1,0.041667
2,198135,ADE,AF,24,1,0.041667
3,198135,BC-,AF,24,2,0.083333
4,198135,BOA,AF,72,3,0.041667


In [87]:
# Seleciona o Top 5 de cada localização separadamente
top5_aq_af = (
    resumo_aq_af
    .query("Localizacao in ['AQ', 'AF']")
    .dropna(subset=["Percentual_Defeito"])
    .sort_values(
        [
            "OP Vertech",
            "Localizacao",
            "Percentual_Defeito",
            "Desvio sigla",
        ],
        ascending=[True, True, False, True],
    )
    .groupby(
        ["OP Vertech", "Localizacao"],
        sort=False,
        observed=True,
    )
    .head(5)
)

# Mantém somente OPs que possuem dados nas duas localizações
ops_com_aq_e_af = (
    top5_aq_af
    .groupby(
        "OP Vertech",
        observed=True,
    )["Localizacao"]
    .nunique()
    .loc[lambda valores: valores.eq(2)]
    .index
)

cores = {
    "AQ": "#D62728",
    "AF": "#1F77B4",
}

for op_vertech in ops_com_aq_e_af:
    dados_op = top5_aq_af[
        top5_aq_af["OP Vertech"].eq(op_vertech)
    ]

    aq = (
        dados_op[
            dados_op["Localizacao"].eq("AQ")
        ]
        .sort_values(
            "Percentual_Defeito",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    af = (
        dados_op[
            dados_op["Localizacao"].eq("AF")
        ]
        .sort_values(
            "Percentual_Defeito",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    # Posições representam o ranking em cada localização
    posicoes_aq = list(range(len(aq)))
    posicoes_af = list(range(len(af)))
    quantidade_linhas = max(len(aq), len(af))

    # Mantém os dois lados com a mesma escala
    maior_percentual = max(
        aq["Percentual_Defeito"].max(),
        af["Percentual_Defeito"].max(),
    )

    limite_eixo = (
        maior_percentual * 1.25
        if maior_percentual > 0
        else 0.01
    )

    fig = make_subplots(
        rows=1,
        cols=2,
        horizontal_spacing=0.10,
        subplot_titles=(
            "<b>Localização AQ</b>",
            "<b>Localização AF</b>",
        ),
    )

    # Top 5 da AQ — barras voltadas para a esquerda
    fig.add_trace(
        go.Bar(
            x=aq["Percentual_Defeito"],
            y=posicoes_aq,
            orientation="h",
            marker_color=cores["AQ"],
            text=aq["Percentual_Defeito"],
            texttemplate="%{text:.2%}",
            textposition="outside",
            cliponaxis=False,
            customdata=aq[
                ["Qtd_Defeito"]
            ].to_numpy(),
            hovertemplate=(
                "<b>Desvio:</b> %{y}<br>"
                "<b>Localização:</b> AQ<br>"
                "<b>Percentual:</b> %{x:.2%}<br>"
                "<b>Qtd. defeitos:</b> %{customdata[0]:,.0f}"
                "<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )

    # Top 5 da AF — barras voltadas para a direita
    fig.add_trace(
        go.Bar(
            x=af["Percentual_Defeito"],
            y=posicoes_af,
            orientation="h",
            marker_color=cores["AF"],
            text=af["Percentual_Defeito"],
            texttemplate="%{text:.2%}",
            textposition="outside",
            cliponaxis=False,
            customdata=af[
                ["Qtd_Defeito"]
            ].to_numpy(),
            hovertemplate=(
                "<b>Desvio:</b> %{y}<br>"
                "<b>Localização:</b> AF<br>"
                "<b>Percentual:</b> %{x:.2%}<br>"
                "<b>Qtd. defeitos:</b> %{customdata[0]:,.0f}"
                "<extra></extra>"
            ),
        ),
        row=1,
        col=2,
    )

    # Eixo da AQ invertido: zero fica próximo ao centro
    fig.update_xaxes(
        title_text="Percentual AQ",
        tickformat=".1%",
        range=[limite_eixo, 0],
        row=1,
        col=1,
    )

    # Eixo da AF: zero fica próximo ao centro
    fig.update_xaxes(
        title_text="Percentual AF",
        tickformat=".1%",
        range=[0, limite_eixo],
        row=1,
        col=2,
    )

    # Desvios próprios da AQ
    fig.update_yaxes(
        title_text="Top 5 desvios da AQ",
        tickmode="array",
        tickvals=posicoes_aq,
        ticktext=aq["Desvio sigla"],
        range=[
            quantidade_linhas - 0.5,
            -0.5,
        ],
        row=1,
        col=1,
    )

    # Desvios próprios da AF, exibidos à direita
    fig.update_yaxes(
        title_text="Top 5 desvios da AF",
        tickmode="array",
        tickvals=posicoes_af,
        ticktext=af["Desvio sigla"],
        range=[
            quantidade_linhas - 0.5,
            -0.5,
        ],
        side="right",
        row=1,
        col=2,
    )

    fig.update_layout(
        title=dict(
            text=(
                "Top 5 desvios detectados em AQ e AF"
                f"<br>OP Vertech: {op_vertech}"
            ),
            x=0.5,
        ),
        template="plotly_white",
        width=950,
        height=480,
        showlegend=False,
        bargap=0.20,
        margin=dict(
            l=100,
            r=100,
            t=110,
            b=65,
        ),
    )

    fig.show()

## Quantidade de defeitos hora a hora

In [88]:
# ============================================================
# 1. Preparação da base horária
# ============================================================

base_horaria_defeitos = df_hh.copy()

# Padroniza as colunas de texto
for coluna in [
    "OP Vertech",
    "Localizacao",
    "Desvio sigla",
]:
    base_horaria_defeitos[coluna] = (
        base_horaria_defeitos[coluna]
        .astype("string")
        .str.strip()
    )

base_horaria_defeitos["Localizacao"] = (
    base_horaria_defeitos["Localizacao"]
    .str.upper()
)

# Garante que as colunas utilizadas nos cálculos
# estejam em formato numérico
for coluna in [
    "Hora a Hora Wht",
    "Qtd Amostra (corrigido)",
    "Qtd Defeito",
]:
    base_horaria_defeitos[coluna] = pd.to_numeric(
        base_horaria_defeitos[coluna],
        errors="coerce",
    )

# Mantém somente as localizações analisadas e
# remove registros sem as informações essenciais
base_horaria_defeitos = (
    base_horaria_defeitos[
        base_horaria_defeitos[
            "Localizacao"
        ].isin(["AQ", "AF", "CF"])
    ]
    .dropna(
        subset=[
            "OP Vertech",
            "Localizacao",
            "Hora a Hora Wht",
        ]
    )
    .copy()
)

# Mantém somente horas válidas
base_horaria_defeitos = (
    base_horaria_defeitos[
        base_horaria_defeitos[
            "Hora a Hora Wht"
        ].between(0, 23)
    ]
    .copy()
)

base_horaria_defeitos["Hora a Hora Wht"] = (
    base_horaria_defeitos[
        "Hora a Hora Wht"
    ].astype(int)
)


# ============================================================
# 2. Consolidação por OP, localização e hora
# ============================================================

resumo_defeitos_hora = (
    base_horaria_defeitos
    .groupby(
        [
            "OP Vertech",
            "Localizacao",
            "Hora a Hora Wht",
        ],
        as_index=False,
        observed=True,
    )
    .agg(
        Quantidade_Defeitos=(
            "Qtd Defeito",
            "sum",
        ),
        Soma_Amostras_por_Desvio=(
            "Qtd Amostra (corrigido)",
            "sum",
        ),
        Quantidade_Desvios_Distintos=(
            "Desvio sigla",
            "nunique",
        ),
    )
)

# Calcula o percentual ponderado, evitando divisão por zero
resumo_defeitos_hora[
    "Percentual_Defeito_Ponderado"
] = (
    resumo_defeitos_hora[
        "Quantidade_Defeitos"
    ]
    / resumo_defeitos_hora[
        "Soma_Amostras_por_Desvio"
    ].where(
        resumo_defeitos_hora[
            "Soma_Amostras_por_Desvio"
        ].gt(0)
    )
)


# ============================================================
# 3. Ordem do dia produtivo e cores
# ============================================================

# Dia produtivo: 07h até 06h
ORDEM_HORAS_DEFEITOS = (
    list(range(7, 24))
    + list(range(0, 7))
)

ROTULOS_HORAS_DEFEITOS = [
    f"{hora:02d}:00"
    for hora in ORDEM_HORAS_DEFEITOS
]

LOCALIZACOES = [
    "AQ",
    "AF",
    "CF",
]

cores_localizacoes = {
    "AQ": "#D62728",
    "AF": "#1F77B4",
    "CF": "#2CA02C",
}


# ============================================================
# 4. Identificação das OPs
# ============================================================

ops_disponiveis = sorted(
    resumo_defeitos_hora[
        "OP Vertech"
    ]
    .dropna()
    .unique()
)

op_inicial = ops_disponiveis[0]


# ============================================================
# 5. Criação do gráfico
# ============================================================

fig = go.Figure()

# Guarda a OP correspondente a cada linha do gráfico
op_por_trace = []

for op_vertech in ops_disponiveis:
    dados_op = resumo_defeitos_hora[
        resumo_defeitos_hora[
            "OP Vertech"
        ].eq(op_vertech)
    ]

    for localizacao in LOCALIZACOES:
        # Seleciona uma OP e localização
        dados_localizacao = (
            dados_op[
                dados_op[
                    "Localizacao"
                ].eq(localizacao)
            ]
            .set_index("Hora a Hora Wht")
            .reindex(ORDEM_HORAS_DEFEITOS)
            .rename_axis("Hora a Hora Wht")
            .reset_index()
        )

        # Adiciona os rótulos das horas
        dados_localizacao["Hora_Producao"] = (
            ROTULOS_HORAS_DEFEITOS
        )

        # Cada trace representa uma localização
        # dentro de uma determinada OP
        fig.add_trace(
            go.Scatter(
                x=dados_localizacao[
                    "Hora_Producao"
                ],
                y=dados_localizacao[
                    "Quantidade_Defeitos"
                ],
                mode="lines+markers",
                name=localizacao,
                legendgroup=localizacao,

                # Inicialmente mostra somente a primeira OP
                visible=(
                    op_vertech == op_inicial
                ),

                # Não conecta horários sem registro
                connectgaps=False,

                line={
                    "width": 3,
                    "color":
                        cores_localizacoes[
                            localizacao
                        ],
                },
                marker={
                    "size": 8,
                    "color":
                        cores_localizacoes[
                            localizacao
                        ],
                },

                # Informações adicionais do hover
                customdata=dados_localizacao[
                    [
                        "Soma_Amostras_por_Desvio",
                        "Quantidade_Desvios_Distintos",
                        "Percentual_Defeito_Ponderado",
                    ]
                ].to_numpy(),

                hovertemplate=(
                    "<b>Localização:</b> "
                    f"{localizacao}<br>"
                    "<b>Hora:</b> %{x}<br>"
                    "<b>Quantidade de defeitos:</b> "
                    "%{y:,.0f}<br>"
                    "<b>Soma das amostras:</b> "
                    "%{customdata[0]:,.0f}<br>"
                    "<b>Desvios distintos:</b> "
                    "%{customdata[1]:,.0f}<br>"
                    "<b>Percentual ponderado:</b> "
                    "%{customdata[2]:.2%}"
                    "<extra></extra>"
                ),
            )
        )

        op_por_trace.append(op_vertech)


# ============================================================
# 6. Dropdown nativo do Plotly para selecionar a OP
# ============================================================

botoes_ops = [
    {
        "label": str(op_vertech),
        "method": "update",
        "args": [
            {
                # Exibe somente as linhas da OP selecionada
                "visible": [
                    op_trace == op_vertech
                    for op_trace in op_por_trace
                ],
            },
            {
                # Atualiza o título
                "title.text": (
                    "Comportamento da quantidade de "
                    "defeitos ao longo do dia"
                    f"<br>OP Vertech: {op_vertech}"
                    "<br><sup>Ausência de ponto indica "
                    "ausência de registro naquela hora"
                    "</sup>"
                ),
            },
        ],
    }
    for op_vertech in ops_disponiveis
]


# ============================================================
# 7. Configuração dos eixos
# ============================================================

fig.update_xaxes(
    title="Hora do dia produtivo",
    type="category",
    categoryorder="array",
    categoryarray=ROTULOS_HORAS_DEFEITOS,
)

fig.update_yaxes(
    title="Quantidade de defeitos",
    rangemode="tozero",
)


# ============================================================
# 8. Layout do gráfico
# ============================================================

fig.update_layout(
    title={
        "text": (
            "Comportamento da quantidade de "
            "defeitos ao longo do dia"
            f"<br>OP Vertech: {op_inicial}"
            "<br><sup>Ausência de ponto indica "
            "ausência de registro naquela hora"
            "</sup>"
        ),
        "x": 0.5,
    },
    template="plotly_white",
    width=1050,
    height=520,
    hovermode="x unified",
    legend_title="Localização",

    # Dropdown interno do Plotly
    updatemenus=[
        {
            "buttons": botoes_ops,
            "direction": "down",
            "showactive": True,
            "active": 0,
            "x": 0,
            "y": 1.18,
            "xanchor": "left",
            "yanchor": "top",
        }
    ],

    # Texto acima do dropdown
    annotations=[
        {
            "text": "<b>OP Vertech:</b>",
            "x": 0,
            "y": 1.26,
            "xref": "paper",
            "yref": "paper",
            "showarrow": False,
            "xanchor": "left",
        }
    ],

    margin={
        "l": 80,
        "r": 50,
        "t": 150,
        "b": 70,
    },
)


# ============================================================
# 9. Exibição única
# ============================================================

fig.show()

# Playground

In [89]:
df_hh

,Data wht (dia) amostra,Forno,Maquina,Prefixo,OP Vertech,NQI (amostra),Hora a Hora Wht,Localizacao,Desvio sigla,Qtd Amostra (corrigido),Qtd Defeito,% defeito
0,2026-08-10,A,A1,SB -1035-SWN,198618,F,7,AQ,BOL,12,1,0.083333
1,2026-08-10,A,A1,SB -1035-SWN,198618,F,7,AF,BOL,24,3,0.125000
2,2026-08-10,A,A1,SB -1035-SWN,198618,F,8,AQ,BOL,12,1,0.083333
3,2026-08-10,A,A1,SB -1035-SWN,198618,F,8,AQ,DOB,12,1,0.083333
4,2026-08-10,A,A1,SB -1035-SWN,198618,F,8,AF,BOL,34,12,0.352941
...,...,...,...,...,...,...,...,...,...,...,...,...
2094,2026-08-10,1,11,LB -0586-S,198660,B,6,AF,ATR,8,1,0.125000
2095,2026-08-10,1,11,LB -0586-S,198660,B,6,AF,FFU,8,2,0.250000
2096,2026-08-10,1,11,LB -0586-S,198660,B,6,AF,RAB,8,1,0.125000
2097,2026-08-10,1,11,LB -0586-S,198660,B,6,CF,FFU,250,12,0.048000
